In [1]:
# =============================================================================
# Korea Trade Data Collector (Final Integrated Version)
# - HS별 1회 작업(내부에서 12개월 window로 자동 분할)
# - YYYYMM 범위 수집 (START_YYYYMM ~ END_YYYYMM)
# - API 제약: 조회기간 1년(12개월) 이내 → 자동 분할 후 concat
# - 안전 파싱 / 재시도 / 세션 풀 / 6자리 HS 요청 대응
# - 월/분기 집계 + YoY + Long format + DB 업로드
# =============================================================================

import re
import time
import logging
import warnings

import requests
import xmltodict
import pandas as pd
import numpy as np
from pandas.tseries.offsets import MonthEnd
from tqdm import tqdm

from sqlalchemy import create_engine, text
from concurrent.futures import ThreadPoolExecutor, as_completed
from requests.adapters import HTTPAdapter

from DATA.stock_invest_function import *  # get_db_host, fetch_table_data 등
from DATA.hs_code_map import HS_CODE_MAP  # get_db_host, fetch_table_data 등
# -----------------------------------------------------------------------------
# 로깅 설정
# -----------------------------------------------------------------------------
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger(__name__)

# -----------------------------------------------------------------------------
# 설정
# -----------------------------------------------------------------------------
SERVICE_KEY = "2o6NG3ixxDgGQ9S4dWUgsMac9WlxfX46%2BJvFRsAlsXQ6xVi6CZewvNJvbHd4S7exkWwt3YWoKSdwvUNb46kSTQ%3D%3D"

START_YYYYMM = "200701"
END_YYYYMM   = "202602"
REGION_NAME  = "전국"

RETRIES = 3
TIMEOUT = 60

MAX_WORKERS   = 8        # 8~10 추천
REQUEST_DELAY = 0.10     # 서버 안정성 위해 0.1~0.2 추천
CHUNK_SIZE    = 1000

DB_INFO = {
    "host": get_db_host(),
    "port": 3307,
    "user": "stox7412",
    "password": "Apt106503!~",
    "database": "investar",
}

# Unique HS Code 리스트 추출
hs_code_list: list = list(HS_CODE_MAP.keys())

# -----------------------------------------------------------------------------
# 유틸
# -----------------------------------------------------------------------------
def norm_yymm(yymm: str) -> str:
    """YYYYMM 6자리 검증"""
    yymm = str(yymm).strip()
    if not re.fullmatch(r"\d{6}", yymm):
        raise ValueError(f"Invalid YYYYMM (must be 6 digits): {yymm}")
    mm = int(yymm[4:6])
    if not (1 <= mm <= 12):
        raise ValueError(f"Invalid month in YYYYMM: {yymm}")
    return yymm

def yymm_to_int(yymm: str) -> int:
    """YYYYMM -> month index (연*12 + (월-1))"""
    yymm = norm_yymm(yymm)
    return int(yymm[:4]) * 12 + (int(yymm[4:6]) - 1)

def int_to_yymm(m: int) -> str:
    """month index -> YYYYMM"""
    yyyy = m // 12
    mm = (m % 12) + 1
    return f"{yyyy:04d}{mm:02d}"

def split_into_12m_windows(start_yymm: str, end_yymm: str):
    """
    [start_yymm, end_yymm] (포함) 범위를
    API 제약(<=12개월)에 맞게 구간 리스트로 분할.
    """
    s = yymm_to_int(start_yymm)
    e = yymm_to_int(end_yymm)
    if e < s:
        raise ValueError("end_yymm must be >= start_yymm")

    windows = []
    cur = s
    while cur <= e:
        win_end = min(cur + 11, e)  # 최대 12개월
        windows.append((int_to_yymm(cur), int_to_yymm(win_end)))
        cur = win_end + 1
    return windows

def parse_items_to_df(response_text: str) -> pd.DataFrame:
    """
    API 응답(XML)을 안전하게 파싱해 items/item을 DF로 반환.
    정상 구조가 아니면 빈 DF.
    """
    try:
        parsed = xmltodict.parse(response_text)
    except Exception:
        return pd.DataFrame()

    if not isinstance(parsed, dict):
        return pd.DataFrame()

    resp = parsed.get("response")
    if not isinstance(resp, dict):
        return pd.DataFrame()

    body = resp.get("body")
    if not isinstance(body, dict):
        return pd.DataFrame()

    items = body.get("items")
    if not isinstance(items, dict):
        return pd.DataFrame()

    item = items.get("item")
    if item is None:
        return pd.DataFrame()

    if isinstance(item, dict):
        item = [item]
    if not isinstance(item, list):
        return pd.DataFrame()

    return pd.DataFrame(item)

def extract_api_header(response_text: str):
    """resultCode/resultMsg 추출 (없으면 None,None)"""
    try:
        parsed = xmltodict.parse(response_text)
        if not isinstance(parsed, dict):
            return None, None
        resp = parsed.get("response")
        if not isinstance(resp, dict):
            return None, None
        header = resp.get("header")
        if not isinstance(header, dict):
            return None, None
        return header.get("resultCode"), header.get("resultMsg")
    except Exception:
        return None, None

def make_session(max_workers: int) -> requests.Session:
    """연결 재사용 + pool 확장"""
    sess = requests.Session()
    adapter = HTTPAdapter(pool_connections=max_workers, pool_maxsize=max_workers)
    sess.mount("https://", adapter)
    sess.mount("http://", adapter)
    return sess

# -----------------------------------------------------------------------------
# 수집 함수 (단일 window)
# -----------------------------------------------------------------------------
def get_country_export_by_item(session: requests.Session,
                               service_key: str,
                               start_yymm: str,
                               end_yymm: str,
                               hs_code_raw) -> pd.DataFrame:
    """
    단일 window(start_yymm~end_yymm) 수집
    - 요청 HS는 6자리로 절단(안정성)
    - 원본 root_hs_code는 그대로 저장
    """
    start_yymm = norm_yymm(start_yymm)
    end_yymm   = norm_yymm(end_yymm)

    hs_code_raw = str(hs_code_raw).strip()
    hs_code_req = hs_code_raw[:6]

    url = (
        "https://apis.data.go.kr/1220000/Itemtrade/getItemtradeList"
        f"?serviceKey={service_key}"
        f"&strtYymm={start_yymm}"
        f"&endYymm={end_yymm}"
        f"&hsSgn={hs_code_req}"
    )

    for attempt in range(RETRIES):
        try:
            r = session.get(url, timeout=TIMEOUT)

            # 재시도 상태코드
            if r.status_code in (429, 502, 503, 504):
                if attempt < RETRIES - 1:
                    backoff = 2 ** attempt
                    logger.warning(f"Retryable {r.status_code} HS {hs_code_raw} ({start_yymm}-{end_yymm}) sleep {backoff}s")
                    time.sleep(backoff)
                    continue
                else:
                    logger.error(f"Retryable {r.status_code} but retries exhausted HS {hs_code_raw} ({start_yymm}-{end_yymm})")
                    return pd.DataFrame()

            r.raise_for_status()

            df = parse_items_to_df(r.text)
            if df.empty:
                code, msg = extract_api_header(r.text)
                if code or msg:
                    logger.warning(f"No/Err data HS {hs_code_raw} (req={hs_code_req}) {start_yymm}-{end_yymm} code={code} msg={msg}")
                else:
                    logger.warning(f"No data HS {hs_code_raw} (req={hs_code_req}) {start_yymm}-{end_yymm} head={r.text[:200]}")
                return pd.DataFrame()

            df["root_hs_code"] = hs_code_raw
            df["hsSgn_req"] = hs_code_req
            df["req_start_yymm"] = start_yymm
            df["req_end_yymm"] = end_yymm

            if REQUEST_DELAY > 0:
                time.sleep(REQUEST_DELAY)

            return df

        except requests.exceptions.RequestException as e:
            if attempt < RETRIES - 1:
                backoff = 2 ** attempt
                logger.warning(f"Attempt {attempt+1}/{RETRIES} failed HS {hs_code_raw} ({start_yymm}-{end_yymm}): {e} | sleep {backoff}s")
                time.sleep(backoff)
                continue
            logger.error(f"All retries failed HS {hs_code_raw} ({start_yymm}-{end_yymm}): {e}")
            return pd.DataFrame()

        except Exception as e:
            logger.error(f"Unexpected error HS {hs_code_raw} ({start_yymm}-{end_yymm}): {e}")
            return pd.DataFrame()

# -----------------------------------------------------------------------------
# 수집 함수 (HS당 여러 window 자동 분할)
# -----------------------------------------------------------------------------
def fetch_hs_multiwindow(session: requests.Session,
                         service_key: str,
                         start_yymm: str,
                         end_yymm: str,
                         hs_code_raw) -> pd.DataFrame:
    """
    HS 하나에 대해 전체 기간을 12개월 window로 쪼개서 수집 후 concat
    """
    windows = split_into_12m_windows(start_yymm, end_yymm)
    dfs = []

    for (s, e) in windows:
        df = get_country_export_by_item(session, service_key, s, e, hs_code_raw)
        if df is not None and not df.empty:
            dfs.append(df)

    if dfs:
        return pd.concat(dfs, ignore_index=True)
    return pd.DataFrame()

# -----------------------------------------------------------------------------
# 전처리/집계
# -----------------------------------------------------------------------------
def process_and_aggregate(df: pd.DataFrame, region_name: str):
    if df.empty:
        return pd.DataFrame(), pd.DataFrame()

    # 총계 제거(있을 때만)
    if "year" in df.columns:
        df = df[df["year"] != "총계"].copy()

    # 날짜 처리
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        if "year" in df.columns:
            df["new_date"] = pd.to_datetime(df["year"].astype(str).str.replace(".", "-"), errors="coerce") + MonthEnd(0)
        else:
            df["new_date"] = pd.NaT

    df = df.dropna(subset=["new_date"]).copy()
    df = df.set_index("new_date")

    df["new_year"] = df.index.year
    df["new_quarter"] = df.index.quarter
    df["new_month"] = df.index.month

    numeric_cols = ["balPayments", "expDlr", "expWgt", "impDlr", "impWgt"]
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0.0)
        else:
            df[col] = 0.0

    base_cols = ["hsCode", "statKor", "root_hs_code", "hsSgn_req"]
    time_cols = ["new_year", "new_quarter", "new_month"]
    keep_cols = [c for c in (base_cols + time_cols + numeric_cols) if c in df.columns]

    clean_df = df[keep_cols].copy()
    clean_df["region"] = region_name

    agg_dict = {"balPayments": "sum", "expDlr": "sum", "impDlr": "sum", "expWgt": "sum", "impWgt": "sum"}

    by_m = (clean_df.groupby(["root_hs_code", "new_year", "new_quarter", "new_month"])
            .agg(agg_dict)
            .reset_index())
    by_m["region"] = region_name

    by_q = (clean_df.groupby(["root_hs_code", "new_year", "new_quarter"])
            .agg(agg_dict)
            .reset_index())
    by_q["region"] = region_name

    return by_q, by_m

def add_yoy_growth(df: pd.DataFrame, steps: int) -> pd.DataFrame:
    if df.empty:
        return df

    out = df.copy()

    if steps == 12:
        out["date"] = pd.to_datetime(out["new_year"].astype(str) + "-" +
                                     out["new_month"].astype(str).str.zfill(2) + "-01") + MonthEnd(0)
    elif steps == 4:
        end_month_map = {1: "03", 2: "06", 3: "09", 4: "12"}
        end_month = out["new_quarter"].map(end_month_map)
        out["date"] = pd.to_datetime(out["new_year"].astype(str) + "-" + end_month + "-01") + MonthEnd(0)
    else:
        raise ValueError("steps는 12(월) 또는 4(분기)만 지원합니다.")

    out = out.sort_values(["root_hs_code", "date"])
    g = out.groupby("root_hs_code")
    out["expDlr_yoy"] = g["expDlr"].transform(lambda x: x.pct_change(periods=steps))
    out["impDlr_yoy"] = g["impDlr"].transform(lambda x: x.pct_change(periods=steps))

    return out

def reshape_to_long(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return df

    id_vars = ["date", "root_hs_code"]
    value_vars = ["balPayments", "expDlr", "impDlr", "expWgt", "impWgt", "expDlr_yoy", "impDlr_yoy"]
    value_vars = [c for c in value_vars if c in df.columns]

    long_df = df.melt(id_vars=id_vars, value_vars=value_vars,
                      var_name="indicator", value_name="value")
    long_df = long_df.dropna(subset=["value"])
    return long_df

# -----------------------------------------------------------------------------
# DB 업로드
# -----------------------------------------------------------------------------
def upload_to_db_optimized(df_long: pd.DataFrame, db_info: dict, table_name: str):
    if df_long.empty:
        logger.info(f"⚠️ 업로드할 데이터가 비어있습니다 ({table_name}).")
        return

    df_long = df_long.copy()
    df_long["date"] = pd.to_datetime(df_long["date"]).dt.date
    df_long = df_long.replace([np.inf, -np.inf], np.nan)
    df_long = df_long.where(pd.notnull(df_long), None)

    engine = create_engine(
        f"mysql+pymysql://{db_info['user']}:{db_info['password']}@{db_info['host']}:{db_info['port']}/{db_info['database']}",
        pool_size=10,
        max_overflow=10,
    )

    create_sql = text(f"""
    CREATE TABLE IF NOT EXISTS {table_name} (
        `date` DATE,
        `root_hs_code` VARCHAR(20),
        `indicator` VARCHAR(50),
        `value` DOUBLE,
        PRIMARY KEY (`date`, `root_hs_code`, `indicator`)
    ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4;
    """)

    with engine.connect() as conn:
        conn.execute(create_sql)
        conn.commit()

    existing_keys = set()
    try:
        q = text(f"SELECT CONCAT(`date`, '|', `root_hs_code`, '|', `indicator`) AS key_combo FROM {table_name}")
        with engine.connect() as conn:
            result = conn.execute(q)
            existing_keys = {row[0] for row in result}
    except Exception as e:
        logger.warning(f"기존 데이터 조회 실패({table_name}): {e}")

    df_long["key_combo"] = (pd.to_datetime(df_long["date"]).dt.strftime("%Y-%m-%d") + "|" +
                            df_long["root_hs_code"].astype(str) + "|" +
                            df_long["indicator"].astype(str))
    df_to_upload = df_long[~df_long["key_combo"].isin(existing_keys)].drop(columns=["key_combo"])

    if df_to_upload.empty:
        logger.info(f"⚠️ 업로드할 새로운 데이터가 없습니다 ({table_name}).")
        return

    logger.info(f"업로드 대상({table_name}): {len(df_to_upload):,}건")

    for i in tqdm(range(0, len(df_to_upload), CHUNK_SIZE), desc=f"DB 업로드({table_name})"):
        chunk = df_to_upload.iloc[i:i + CHUNK_SIZE]
        try:
            chunk.to_sql(name=table_name, con=engine, if_exists="append", index=False, method="multi")
        except Exception as e:
            logger.error(f"청크 {i//CHUNK_SIZE + 1} 업로드 실패({table_name}): {e}")

    logger.info(f"✅ 총 {len(df_to_upload):,}건 업로드 완료 ({table_name})")

# =============================================================================
# 메인
# =============================================================================
def main():
    t0 = time.time()

    # YYYYMM 검증
    norm_yymm(START_YYYYMM)
    norm_yymm(END_YYYYMM)

    windows = split_into_12m_windows(START_YYYYMM, END_YYYYMM)
    logger.info(f"📅 전체 수집기간: {START_YYYYMM}~{END_YYYYMM} | window 개수: {len(windows)} | windows={windows}")

    # HS 코드 로드
    # hs_data = fetch_table_data(DB_INFO, "target_hs_code")
    # HS_CODES = hs_data["hs_code"].dropna().astype(str).unique().tolist()

    HS_CODES : list = list(HS_CODE_MAP.keys())


    logger.info("🚀 무역 데이터 수집 시작")
    logger.info(f"📊 HS 코드 수: {len(HS_CODES)}")
    logger.info(f"⚡ 동시 워커: {MAX_WORKERS}")

    session = make_session(MAX_WORKERS)

    all_dfs = []
    errors = []
    empty_count = 0

    try:
        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
            future_map = {
                executor.submit(fetch_hs_multiwindow, session, SERVICE_KEY, START_YYYYMM, END_YYYYMM, hs): hs
                for hs in HS_CODES
            }

            for future in tqdm(as_completed(future_map), total=len(future_map), desc="API 요청"):
                hs = future_map[future]
                try:
                    df = future.result(timeout=300)
                    if df is not None and not df.empty:
                        all_dfs.append(df)
                    else:
                        empty_count += 1
                except Exception as e:
                    logger.error(f"HS {hs} 처리 실패: {e}")
                    errors.append(hs)

    finally:
        session.close()

    logger.info(f"⏱️ 수집 종료. 소요: {time.time()-t0:.2f}초 | 실패 HS: {len(errors)} | 빈결과 HS: {empty_count}")

    if not all_dfs:
        logger.error("❌ 수집된 데이터가 없습니다.")
        return None

    final_df = pd.concat(all_dfs, ignore_index=True)
    logger.info(f"📋 총 수집 레코드: {len(final_df):,}건")

    processed_q, processed_m = process_and_aggregate(final_df, REGION_NAME)
    logger.info(f"📊 월별 집계: {len(processed_m):,}건 | 분기별 집계: {len(processed_q):,}건")

    if processed_m.empty:
        logger.warning("⚠️ 처리할 월별 데이터가 없습니다.")
        return None

    monthly_with_yoy = add_yoy_growth(processed_m, steps=12)
    quarterly_with_yoy = add_yoy_growth(processed_q, steps=4)

    monthly_long = reshape_to_long(monthly_with_yoy)
    quarterly_long = reshape_to_long(quarterly_with_yoy)

    logger.info(f"📌 월별 long: {len(monthly_long):,}건 | 분기별 long: {len(quarterly_long):,}건")

    ans = input("🗃️ 데이터베이스에 업로드하시겠습니까? (y/n): ").strip().lower()
    if ans == "y":
        upload_to_db_optimized(monthly_long, DB_INFO, "korea_monthly_trade_data")
        upload_to_db_optimized(quarterly_long, DB_INFO, "korea_quarterly_trade_data")

    logger.info("✅ 전체 파이프라인 완료")

    return {
        "final_raw": final_df,
        "monthly": monthly_with_yoy,
        "quarterly": quarterly_with_yoy,
        "monthly_long": monthly_long,
        "quarterly_long": quarterly_long,
        "errors": errors,
        "empty_hs_count": empty_count,
    }

# 실행
out = main()

# 결과 변수 편의 할당
if out:
    final_raw = out["final_raw"]
    monthly_trade_data = out["monthly"]
    quarterly_trade_data = out["quarterly"]
    monthly_long_data = out["monthly_long"]
    quarterly_long_data = out["quarterly_long"]
    error_list = out["errors"]

    print("\n💡 사용 가능한 변수들:")
    print("- final_raw")
    print("- monthly_trade_data")
    print("- quarterly_trade_data")
    print("- monthly_long_data")
    print("- quarterly_long_data")
    print("- error_list")

# -----------------------------------------------------------------------------
# 시각화 함수(유지)
# -----------------------------------------------------------------------------
def plot_column_by_hscode(df, hs_code, col_name, start_date=None, end_date=None):
    import matplotlib.pyplot as plt

    if df is None or df.empty:
        print("❌ DataFrame이 비어 있습니다.")
        return
    if "date" not in df.columns:
        print("❌ 'date' 컬럼이 없습니다. add_yoy_growth()를 먼저 실행하세요.")
        return

    hs_code = str(hs_code)
    target_df = df[df["root_hs_code"].astype(str) == hs_code].sort_values("date")

    if target_df.empty:
        print(f"⚠️ root_hs_code {hs_code}에 해당하는 데이터가 없습니다.")
        return

    if col_name not in target_df.columns:
        print(f"❌ '{col_name}' 컬럼이 DataFrame에 없습니다.")
        return

    if start_date:
        target_df = target_df[target_df["date"] >= pd.to_datetime(start_date)]
    if end_date:
        target_df = target_df[target_df["date"] <= pd.to_datetime(end_date)]

    if target_df.empty:
        print("⚠️ 지정한 날짜 범위에 데이터가 없습니다.")
        return

    plt.figure(figsize=(12, 6))
    plt.plot(target_df["date"], target_df[col_name], marker="o", label=col_name)

    last_x = target_df["date"].iloc[-1]
    last_y = target_df[col_name].iloc[-1]

    if "yoy" in col_name:
        plt.text(last_x, last_y, f"{last_y * 100:,.2f}%", fontsize=12, ha="left", va="bottom", color="red")
    else:
        plt.text(last_x, last_y, f"{last_y:,.0f}", fontsize=12, ha="left", va="bottom", color="red")

    plt.title(f"{col_name} 추이 (root_hs_code: {hs_code})")
    plt.xlabel("Date")
    plt.ylabel(col_name)
    plt.legend()
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

2026-03-15 15:57:01,660 - INFO - 📅 전체 수집기간: 200701~202602 | window 개수: 20 | windows=[('200701', '200712'), ('200801', '200812'), ('200901', '200912'), ('201001', '201012'), ('201101', '201112'), ('201201', '201212'), ('201301', '201312'), ('201401', '201412'), ('201501', '201512'), ('201601', '201612'), ('201701', '201712'), ('201801', '201812'), ('201901', '201912'), ('202001', '202012'), ('202101', '202112'), ('202201', '202212'), ('202301', '202312'), ('202401', '202412'), ('202501', '202512'), ('202601', '202602')]
2026-03-15 15:57:01,662 - INFO - 🚀 무역 데이터 수집 시작
2026-03-15 15:57:01,663 - INFO - 📊 HS 코드 수: 495
2026-03-15 15:57:01,663 - INFO - ⚡ 동시 워커: 8
API 요청:   0%|          | 0/495 [00:00<?, ?it/s]2026-03-15 15:57:03,966 - WARNING - No/Err data HS 870340 (req=870340) 200701-200712 code=00 msg=정상서비스.
2026-03-15 15:57:05,957 - WARNING - No/Err data HS 870340 (req=870340) 200801-200812 code=00 msg=정상서비스.
2026-03-15 15:57:08,113 - WARNING - No/Err data HS 870340 (req=870340) 200901-20


💡 사용 가능한 변수들:
- final_raw
- monthly_trade_data
- quarterly_trade_data
- monthly_long_data
- quarterly_long_data
- error_list


In [2]:
monthly_df = out['monthly']
# monthly_df[monthly_df['root_hs_code'] == '854232'].tail(13)

In [4]:
# date 인덱스 생성
monthly_df['date'] = pd.to_datetime(
    monthly_df['new_year'].astype(str) + '-' + monthly_df['new_month'].astype(str).str.zfill(2)
)

# pivot table 생성
pivot_df = monthly_df.pivot_table(
    index='date',
    columns='root_hs_code',
    values='expDlr',
    aggfunc='sum'
)

pivot_df.index = pd.DatetimeIndex(pivot_df.index).to_period('M')

print(pivot_df)

root_hs_code      030354      030389      030487      030633      121221  \
date                                                                       
2007-01              NaN         NaN         NaN         NaN         NaN   
2007-02              NaN         NaN         NaN         NaN         NaN   
2007-03              NaN         NaN         NaN         NaN         NaN   
2007-04              NaN         NaN         NaN         NaN         NaN   
2007-05              NaN         NaN         NaN         NaN         NaN   
...                  ...         ...         ...         ...         ...   
2025-10       12988226.0   3900999.0  20514094.0   9097348.0  31603699.0   
2025-11       18508698.0   7600561.0  22436329.0   9299194.0  30195425.0   
2025-12       54414965.0   9042035.0  22134387.0  13243820.0  34928150.0   
2026-01       32386717.0  11555723.0  25513365.0  10648201.0  50874596.0   
2026-02       40681886.0   8318806.0  19871006.0  11789529.0  37894159.0   

root_hs_cod

In [9]:
# pivot_df.to_csv(r'C:\Users\82108\OneDrive\INVESTMENT\Data_Asnal\Export_data\export_202602.csv')